In [19]:
import sys
import json
import requests
import pandas as pd

WEBSITE_API = "https://rest.uniprot.org/"

def get_url(url, **kwargs):
    """Fetch a URL and return the response, exiting on an HTTP error."""
    response = requests.get(url, **kwargs)

    if not response.ok:
        print(response.text)
        response.raise_for_status()
        sys.exit()

    return response

# 1. Data layout

# Surfactant protein D ("Sftpd") is the knocked-out gene, so it is left out when
# ranking genes in the knockout comparisons.

CSV_PATH = "NewAffydata.csv"          

LABEL_COL = 0                          # first column holds the condition labels
EXCLUDE_NAMES = {"sftpd"}              # surfactant protein D (knocked out), skipped when ranking
TOP_N = 3                              # number of top genes to report per condition

raw = pd.read_csv(CSV_PATH, header=None, dtype=str)

DATA_COLS = list(range(1, raw.shape[1]))

gene_names = raw.loc[0, DATA_COLS].tolist()

# 2. Map each gene to a single column

# Several genes appear in more than one probe column; the first column for each
# gene is used. A combined-probe label such as "Ear1 /// Ear12 /// Ear2 /// Ear3"
# maps one probe to several genes, and the first symbol before "///" is used for
# the UniProt search.
def search_symbol(raw_name):
    """First symbol before any '///' in a combined-probe label."""
    return raw_name.split("///")[0].strip()

gene_positions = {}                    # gene symbol -> its first data column
gene_query = {}                        # gene symbol -> symbol used for the UniProt search
for pos, name in enumerate(gene_names):
    clean = str(name).strip()
    if clean in gene_positions:
        continue                       # keep only the first column for this gene
    gene_positions[clean] = DATA_COLS[pos]
    gene_query[clean] = search_symbol(clean)

# 3. Read each condition row into a per-gene value

def row_values(line):
    """Value of each gene's column for one row of the file."""
    numeric = pd.to_numeric(raw.loc[line, DATA_COLS], errors="coerce")
    return {gene: numeric[col] for gene, col in gene_positions.items()}

row_label = {}                          # row index -> condition label
row_genes = {}                          # row index -> {gene: value}
for line in range(1, raw.shape[0]):     # every row after the header row
    label = raw.loc[line, LABEL_COL]
    if pd.isna(label):
        continue
    row_label[line] = str(label).strip()
    row_genes[line] = row_values(line)

# 4. Split each label into (strain, condition)

STRAINS = ["BL/6", "Balb/c", "SP-D KO"]

def split_label(label):
    for strain in STRAINS:
        if label.startswith(strain):
            return strain, label[len(strain):].strip()
    return None, label

# For each strain, map its conditions to their row indices.
rows_by_strain = {s: {} for s in STRAINS}
for line, label in row_label.items():
    strain, cond = split_label(label)
    if strain is not None:
        rows_by_strain[strain][cond] = line

# 5. Look up a gene symbol on UniProt (Mus musculus)

_uniprot_cache = {}

def lookup_uniprot(symbol):
    """Return (accession, entry URL) for a mouse gene symbol, caching results."""
    if symbol in _uniprot_cache:
        return _uniprot_cache[symbol]

    # gene:<symbol> is an indexed lookup; size=1 returns only the best hit.
    query_url = (
        f"{WEBSITE_API}uniprotkb/search"
        f"?query=gene:{symbol}+AND+organism_id:10090&fields=accession&format=json&size=1"
    )
    r = get_url(query_url)
    results = r.json()["results"]

    if results:
        accession = results[0]["primaryAccession"]
        entry_url = f"https://www.uniprot.org/uniprotkb/{accession}/entry"
    else:
        accession, entry_url = "NA", "NA"

    _uniprot_cache[symbol] = (accession, entry_url)
    return accession, entry_url

# 6. Build a comparison table for two strains

def build_table(strain_a, strain_b, exclude_spd=True):
    """Compare strain_a with strain_b; the difference is strain_b minus strain_a.

    When exclude_spd is True, surfactant protein D is left out of the ranking.
    """
    a_rows = rows_by_strain[strain_a]
    b_rows = rows_by_strain[strain_b]
    conditions = [c for c in a_rows if c in b_rows]   # conditions shared by both strains

    diff_header = f"Difference ([{strain_b}] - [{strain_a}])"

    records = []
    for cond in conditions:
        a_vals = row_genes[a_rows[cond]]
        b_vals = row_genes[b_rows[cond]]

        # Difference for each gene, as strain_b minus strain_a.
        diffs = {
            gene: b_vals[gene] - a_vals[gene]
            for gene in gene_positions
            if not (exclude_spd and gene.lower() in EXCLUDE_NAMES)
        }

        # Take the TOP_N genes with the largest absolute difference.
        working = dict(diffs)
        top_genes = []
        for _ in range(TOP_N):
            gene = max(working, key=lambda g: abs(working[g]))
            top_genes.append(gene)
            del working[gene]                          # remove it so the next pass finds the next largest

        gene_labels, differences, accessions, links = [], [], [], []
        for gene in top_genes:
            gene_labels.append(gene)
            differences.append(round(float(diffs[gene]), 3))
            accession, entry_url = lookup_uniprot(gene_query[gene])
            accessions.append(accession)
            links.append(entry_url)

        records.append(
            {
                "Condition": row_label[a_rows[cond]],
                "Gene Names": gene_labels,
                diff_header: differences,
                "Accession Numbers": accessions,
                "UniProt Links": links,
            }
        )

    return pd.DataFrame(records)

# 7. Build and display the three tables

table_bl6 = build_table("BL/6", "SP-D KO")
table_balb = build_table("Balb/c", "SP-D KO")
table_wt = build_table("BL/6", "Balb/c", exclude_spd=False)

from IPython.display import display

# Show full cell contents so the UniProt links are not truncated.
pd.set_option("display.max_colwidth", None)

def show(df, title):
    """Display a table with left-aligned columns."""
    print(title)
    try:
        display(df.style.set_properties(**{"text-align": "left"})
                        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
    except Exception:
        print(df.to_string(index=False, justify="left"))

show(table_bl6, "BL/6 vs SP-D KO  (top 3 genes per condition)")
show(table_balb, "Balb/c vs SP-D KO  (top 3 genes per condition)")
show(table_wt, "BL/6 vs Balb/c  (top 3 genes per condition)")

BL/6 vs SP-D KO  (top 3 genes per condition)


,Condition,Gene Names,Difference ([SP-D KO] - [BL/6]),Accession Numbers,UniProt Links
0,BL/6 air,"['Ear1', 'Ear3', 'Srrm2']","[1779.0, 489.0, 447.0]","['P97426', 'O35290', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/O35290/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
1,BL/6 24h Af+air,"['Nos1ap', 'Prmt2', 'C2']","[-1904.0, -1429.0, -807.0]","['B7ZNH1', 'F6Y3M9', 'B8JJM9']","['https://www.uniprot.org/uniprotkb/B7ZNH1/entry', 'https://www.uniprot.org/uniprotkb/F6Y3M9/entry', 'https://www.uniprot.org/uniprotkb/B8JJM9/entry']"
2,BL/6 168h Af+air,"['Avpi1', 'C1qa', 'Ear1']","[3461.0, 3207.0, 2149.0]","['E0CXY9', 'Q3TXB1', 'P97426']","['https://www.uniprot.org/uniprotkb/E0CXY9/entry', 'https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry']"
3,BL/6 O3,"['Ear1', 'C2', 'Srrm2']","[2498.0, -1120.0, 892.0]","['P97426', 'B8JJM9', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/B8JJM9/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
4,BL/6 24h Af+O3,"['Ear1', 'C1qc', 'Avpi1']","[-2042.0, 1484.0, -1440.0]","['P97426', 'Q02105', 'E0CXY9']","['https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/Q02105/entry', 'https://www.uniprot.org/uniprotkb/E0CXY9/entry']"
5,BL/6 168h Af+O3,"['C1qa', 'C2', 'Ptpla']","[7415.0, 2565.0, -2171.0]","['Q3TXB1', 'B8JJM9', 'Q3V4A5']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/B8JJM9/entry', 'https://www.uniprot.org/uniprotkb/Q3V4A5/entry']"


Balb/c vs SP-D KO  (top 3 genes per condition)


,Condition,Gene Names,Difference ([SP-D KO] - [Balb/c]),Accession Numbers,UniProt Links
0,Balb/c air,"['Ear1', 'Padi2', 'Ear3']","[1750.0, -592.0, 478.0]","['P97426', 'A3KME9', 'O35290']","['https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/A3KME9/entry', 'https://www.uniprot.org/uniprotkb/O35290/entry']"
1,Balb/c 24h Af+air,"['C1qa', 'Ptpla', 'Srrm2']","[-3526.0, -2580.0, -2425.0]","['Q3TXB1', 'Q3V4A5', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/Q3V4A5/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
2,Balb/c 168h Af+air,"['Avpi1', 'Ear1', 'C1qa']","[3239.0, 1768.0, 1443.0]","['E0CXY9', 'P97426', 'Q3TXB1']","['https://www.uniprot.org/uniprotkb/E0CXY9/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/Q3TXB1/entry']"
3,Balb/c O3,"['C1qa', 'Ear1', 'Srrm2']","[-3486.0, 2536.0, 1401.0]","['Q3TXB1', 'P97426', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
4,Balb/c 24h Af+O3,"['C2', 'C1s /// LOC100044326', 'C1qa']","[-2870.0, -2123.0, -1930.0]","['B8JJM9', 'Q14DT6', 'Q3TXB1']","['https://www.uniprot.org/uniprotkb/B8JJM9/entry', 'https://www.uniprot.org/uniprotkb/Q14DT6/entry', 'https://www.uniprot.org/uniprotkb/Q3TXB1/entry']"
5,Balb/c 168h Af+O3,"['C1qa', 'Nos1ap', 'Ear1']","[5143.0, 2305.0, 1796.0]","['Q3TXB1', 'B7ZNH1', 'P97426']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/B7ZNH1/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry']"


BL/6 vs Balb/c  (top 3 genes per condition)


,Condition,Gene Names,Difference ([Balb/c] - [BL/6]),Accession Numbers,UniProt Links
0,BL/6 air,"['Padi2', 'C1qa', 'Srrm2']","[499.0, 494.0, 468.0]","['A3KME9', 'Q3TXB1', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/A3KME9/entry', 'https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
1,BL/6 24h Af+air,"['C1qa', 'Srrm2', 'Ptpla']","[3643.0, 2455.0, 2210.0]","['Q3TXB1', 'A0A087WPS9', 'Q3V4A5']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry', 'https://www.uniprot.org/uniprotkb/Q3V4A5/entry']"
2,BL/6 168h Af+air,"['C1qa', 'Cfd', 'C1qc']","[1764.0, -866.0, 820.0]","['Q3TXB1', 'B7ZNS9', 'Q02105']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/B7ZNS9/entry', 'https://www.uniprot.org/uniprotkb/Q02105/entry']"
3,BL/6 O3,"['C1qa', 'C2', 'Padi2']","[3263.0, -864.0, 679.0]","['Q3TXB1', 'B8JJM9', 'A3KME9']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/B8JJM9/entry', 'https://www.uniprot.org/uniprotkb/A3KME9/entry']"
4,BL/6 24h Af+O3,"['C1qa', 'Ear1', 'C2']","[3094.0, -3001.0, 2728.0]","['Q3TXB1', 'P97426', 'B8JJM9']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/B8JJM9/entry']"
5,BL/6 168h Af+O3,"['Ptpla', 'C1qa', 'Prmt2']","[-2565.0, 2272.0, -1553.0]","['Q3V4A5', 'Q3TXB1', 'F6Y3M9']","['https://www.uniprot.org/uniprotkb/Q3V4A5/entry', 'https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/F6Y3M9/entry']"
